<a href="https://colab.research.google.com/github/CodeBlue0001/Agrimind/blob/main/state_wise_crop_yield.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
# importing the state wise crop yeild data
state_crop_df = pd.read_csv(r"/content/state_wise_crop_yild.csv")
state_crop_df.head()
#


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,Castor seed,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238333
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909


In [6]:
# state wise rainfal vs crop prediction model
from sklearn.model_selection import train_test_split
# from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report




In [ ]:
# Define features (X) and target (y)
X = state_crop_df.drop('Yield', axis=1)
y = state_crop_df['Yield']

# Identify categorical and numerical features
categorical_features = ['Crop', 'Season', 'State']
numerical_features = ['Crop_Year', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide']

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numerical_features)
    ])

# Import RandomForestRegressor (since Yield is continuous, it's a regression problem)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Create the model pipeline
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('regressor', RandomForestRegressor(random_state=20))])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20)

# Train the model
print("Training the model...")
model_pipeline.fit(X_train, y_train)
print("Model training complete.")

# Make predictions
y_pred = model_pipeline.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Evaluation:")
print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared: {r2:.2f}")

# Display a few predictions vs actual values
results_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
print("\nFirst 10 Actual vs Predicted Yields:")
print(results_df.head(10))

Training the model...


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a scatter plot of Actual vs. Predicted Yields
plt.figure(figsize=(10, 7))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2) # Line for perfect prediction
plt.xlabel('Actual Yield')
plt.ylabel('Predicted Yield')
plt.title('Overall Model: Actual vs. Predicted Crop Yields')
plt.grid(True)
plt.show()

print("\n--- Interpreting the graph ---")
print("The scatter plot shows actual crop yields on the x-axis and predicted yields on the y-axis.")
print("The red dashed line represents a perfect prediction (where Actual = Predicted).")
print("Points close to the red line indicate accurate predictions. Deviations from the line represent prediction errors.")
print(f"Our overall model achieved an R-squared of {r2:.2f}, indicating {r2*100:.0f}% of the variance in yield is explained by the model. The closer the points cluster around the red line, the better the model's performance.")

# Next steps to reduce error and provide more accurate values:
print("\n--- Next Steps: Updating the Model for More Accuracy ---")
print("To further reduce the error and provide more accurate values for the *overall* model, we can perform hyperparameter tuning.")
print("This involves systematically searching for the best combination of parameters for our RandomForestRegressor across the entire dataset (not just state-specific).")
print("I can use `GridSearchCV` or `RandomizedSearchCV` on the `model_pipeline` to find these optimal parameters.")
print("Would you like me to proceed with hyperparameter tuning for the overall model?")

In [ ]:
# The RandomizedSearchCV for overall model tuning was interrupted.
# Please re-run the cell above to complete the hyperparameter tuning if you wish.

### Training State-Wise Specific Models
To achieve more precise predictions by capturing state-specific patterns, I will now train a separate Random Forest Regressor model for each unique state in the dataset. Each model will be trained only on the data pertinent to that specific state.


In [ ]:
# Initialize dictionaries to store models and evaluation results for each state
state_models = {}
state_metrics = {}

# Get unique states from the dataset
unique_states = state_crop_df['State'].unique()

print(f"Starting to train models for {len(unique_states)} unique states...")

for state in unique_states:
    print(f"\n--- Training model for {state} ---")

    # Filter data for the current state
    state_df = state_crop_df[state_crop_df['State'] == state].copy()

    # Skip states with insufficient data for a meaningful train/test split
    # A minimum of 5 samples is set as a heuristic to allow for at least 1 test sample and 4 training samples.
    if len(state_df) < 5:
        print(f"  Skipping {state}: Insufficient data ({len(state_df)} samples). Requires at least 5 samples.")
        continue

    # Define features (X) and target (y) for the current state
    # The 'State' column is constant for the filtered DataFrame, so it's dropped from features.
    X_state = state_df.drop(['Yield', 'State'], axis=1)
    y_state = state_df['Yield']

    # Identify categorical and numerical features for this state (excluding 'State')
    categorical_features_state = ['Crop', 'Season']
    numerical_features_state = ['Crop_Year', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide']

    # Ensure only existing columns are passed to the ColumnTransformer
    current_categorical_features = [f for f in categorical_features_state if f in X_state.columns]
    current_numerical_features = [f for f in numerical_features_state if f in X_state.columns]

    # Check if there are any features left to train on
    if not current_categorical_features and not current_numerical_features:
        print(f"  Skipping {state}: No valid features found after filtering relevant columns.")
        continue

    # Create a column transformer specific to this state's data
    # OneHotEncoder handles unseen categories gracefully with handle_unknown='ignore'
    state_preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), current_categorical_features),
            ('num', 'passthrough', current_numerical_features)
        ],
        remainder='drop' # Drop any columns not explicitly specified
    )

    # Create the model pipeline for the current state
    state_model_pipeline = Pipeline(steps=[
        ('preprocessor', state_preprocessor),
        ('regressor', RandomForestRegressor(random_state=42))
    ])

    # Split the data for the current state into training and testing sets
    try:
        X_train_state, X_test_state, y_train_state, y_test_state = train_test_split(
            X_state, y_state, test_size=0.2, random_state=42
        )
    except ValueError as e:
        print(f"  Skipping {state}: Could not split data ({e}). This usually happens with too few samples or target values.")
        continue

    # Train the model for the current state
    try:
        state_model_pipeline.fit(X_train_state, y_train_state)
    except Exception as e: # Catch potential errors during model fitting
        print(f"  Skipping {state}: Error during model training ({e}).")
        continue

    # Make predictions using the state-specific model
    y_pred_state = state_model_pipeline.predict(X_test_state)

    # Evaluate the model's performance for the current state
    mse_state = mean_squared_error(y_test_state, y_pred_state)
    r2_state = r2_score(y_test_state, y_pred_state)

    print(f"  Mean Squared Error (MSE): {mse_state:.2f}")
    print(f"  R-squared (R2): {r2_state:.2f}")

    # Store the trained model and its evaluation metrics
    state_models[state] = state_model_pipeline
    state_metrics[state] = {'mse': mse_state, 'r2': r2_state}

print("\n--- Summary of State-Wise Model Performance ---")
if not state_metrics:
    print("No models were successfully trained for any state due to insufficient data or errors.")
else:
    # Sort states by R-squared for better readability (highest R2 first)
    sorted_states_metrics = sorted(state_metrics.items(), key=lambda item: item[1]['r2'], reverse=True)
    for state, metrics in sorted_states_metrics:
        print(f"State: {state:<20}, MSE: {metrics['mse']:.2f}, R-squared: {metrics['r2']:.2f}")

print("\nFinished training state-wise models.")


### Saving the State-Wise Models
I will now save each of the trained `RandomForestRegressor` models for each state to a `.pkl` file. This allows you to load and use these models later without retraining them.

In [ ]:
import pickle
import os

# Create a directory to save models if it doesn't exist
model_save_dir = 'state_crop_yield_models'
os.makedirs(model_save_dir, exist_ok=True)

print(f"Saving models to directory: {model_save_dir}/")

for state, model_pipeline in state_models.items():
    # Sanitize state name for use in filename (replace spaces and special chars)
    filename = os.path.join(model_save_dir, f'{state.replace(" ", "_").lower()}_crop_yield_model.pkl')
    try:
        with open(filename, 'wb') as f:
            pickle.dump(model_pipeline, f)
        print(f"  Saved model for {state} to {filename}")
    except Exception as e:
        print(f"  Error saving model for {state}: {e}")

print("\nAll state-wise models saved.")

### Fine-tuning Models for Improved Performance
For states where the initial model performance was suboptimal (e.g., negative or low R-squared), fine-tuning the model's hyperparameters can lead to significant improvements. I will now demonstrate how to fine-tune a model using `GridSearchCV` for the state of Jharkhand, which had a negative R-squared, indicating a very poor initial fit.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Choose a state to fine-tune (e.g., Jharkhand due to its negative R-squared)
state_to_tune = 'Jharkhand'
print(f"\n--- Fine-tuning model for {state_to_tune} ---")

# Retrieve the original data for the state
state_df_tune = state_crop_df[state_crop_df['State'] == state_to_tune].copy()

# Define features (X) and target (y) for the state
X_state_tune = state_df_tune.drop(['Yield', 'State'], axis=1)
y_state_tune = state_df_tune['Yield']

# Identify categorical and numerical features for this state (excluding 'State')
categorical_features_state_tune = ['Crop', 'Season']
numerical_features_state_tune = ['Crop_Year', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide']

# Ensure only existing columns are passed to the ColumnTransformer
current_categorical_features_tune = [f for f in categorical_features_state_tune if f in X_state_tune.columns]
current_numerical_features_tune = [f for f in numerical_features_state_tune if f in X_state_tune.columns]

# Create a column transformer specific to this state's data
state_preprocessor_tune = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), current_categorical_features_tune),
        ('num', 'passthrough', current_numerical_features_tune)
    ],
    remainder='drop' # Drop any columns not explicitly specified
)

# Create a new pipeline for tuning, using the same preprocessor structure
pipeline_for_tuning = Pipeline(steps=[
    ('preprocessor', state_preprocessor_tune),
    ('regressor', RandomForestRegressor(random_state=42)) # Start with a default regressor
])

# Split the data for the current state into training and testing sets
X_train_tune, X_test_tune, y_train_tune, y_test_tune = train_test_split(
    X_state_tune, y_state_tune, test_size=0.2, random_state=42
)

# Define the parameter grid for GridSearchCV
# We prefix parameters with 'regressor__' because we are tuning the 'regressor' step in the pipeline
param_grid = {
    'regressor__n_estimators': [50, 100, 200], # Number of trees in the forest
    'regressor__max_depth': [None, 10, 20],   # Maximum depth of the tree
    'regressor__min_samples_split': [2, 5]   # Minimum number of samples required to split an internal node
}

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=pipeline_for_tuning, param_grid=param_grid,
                           cv=3, n_jobs=-1, verbose=2, scoring='r2')

# Perform the grid search on the training data
print("  Starting Grid Search...")
grid_search.fit(X_train_tune, y_train_tune)
print("  Grid Search complete.")

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print(f"  Best parameters found: {best_params}")
print(f"  Best R-squared from cross-validation: {best_score:.2f}")

# Evaluate the best model on the test set
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test_tune)

mse_tuned = mean_squared_error(y_test_tune, y_pred_tuned)
r2_tuned = r2_score(y_test_tune, y_pred_tuned)

print(f"  Fine-tuned Model Evaluation on Test Set:")
print(f"  Mean Squared Error (MSE): {mse_tuned:.2f}")
print(f"  R-squared (R2): {r2_tuned:.2f}")

# Optionally, update the state_models dictionary with the fine-tuned model
state_models[state_to_tune] = best_model
state_metrics[state_to_tune] = {'mse': mse_tuned, 'r2': r2_tuned}
print(f"  Model for {state_to_tune} updated in state_models with fine-tuned version.")

### Comparing Performance with Gradient Boosting on Jharkhand Data
Since the Random Forest model's performance on Jharkhand was suboptimal, let's explore another ensemble method: Gradient Boosting. I will train a `GradientBoostingRegressor` on the same Jharkhand data and compare its results.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

print(f"\n--- Training GradientBoostingRegressor for {state_to_tune} ---")

# Create a new pipeline for GradientBoostingRegressor, using the same preprocessor structure
pipeline_gb = Pipeline(steps=[
    ('preprocessor', state_preprocessor_tune),
    ('regressor', GradientBoostingRegressor(random_state=42)) # Using GradientBoostingRegressor
])

# Train the GradientBoostingRegressor model for the current state
pipeline_gb.fit(X_train_tune, y_train_tune)

# Make predictions using the GradientBoostingRegressor
y_pred_gb = pipeline_gb.predict(X_test_tune)

# Evaluate the GradientBoostingRegressor model's performance
mse_gb = mean_squared_error(y_test_tune, y_pred_gb)
r2_gb = r2_score(y_test_tune, y_pred_gb)

print(f"  Gradient Boosting Model Evaluation on Test Set for {state_to_tune}:")
print(f"  Mean Squared Error (MSE): {mse_gb:.2f}")
print(f"  R-squared (R2): {r2_gb:.2f}")

# Compare with RandomForestRegressor (fine-tuned)
print(f"\n  RandomForestRegressor (Fine-tuned) for {state_to_tune}:")
print(f"  Mean Squared Error (MSE): {state_metrics[state_to_tune]['mse']:.2f}")
print(f"  R-squared (R2): {state_metrics[state_to_tune]['r2']:.2f}")

# Compare with RandomForestRegressor (initial)
print(f"\n  RandomForestRegressor (Initial) for {state_to_tune}:")
# Retrieve initial metrics for Jharkhand from `state_metrics` before fine-tuning.
# This would typically require storing both initial and fine-tuned metrics separately.
# For this demonstration, we'll assume the initial value was the one before any tuning step.
# As state_metrics was updated, let's re-calculate initial R2 for comparison.

initial_rfr = RandomForestRegressor(random_state=42)
initial_pipeline = Pipeline(steps=[
    ('preprocessor', state_preprocessor_tune),
    ('regressor', initial_rfr)
])
initial_pipeline.fit(X_train_tune, y_train_tune)
y_pred_initial_rfr = initial_pipeline.predict(X_test_tune)
mse_initial_rfr = mean_squared_error(y_test_tune, y_pred_initial_rfr)
r2_initial_rfr = r2_score(y_test_tune, y_pred_initial_rfr)

print(f"  Mean Squared Error (MSE): {mse_initial_rfr:.2f}")
print(f"  R-squared (R2): {r2_initial_rfr:.2f}")


### Creating an Inference Interface for the Overall Model

Now, let's create a function that allows you to easily get predictions from the overall trained model. This function will take the necessary input features for a single prediction and return the estimated crop yield.

In [ ]:
def predict_yield_overall_model(crop, crop_year, season, state, area, production, annual_rainfall, fertilizer, pesticide):
    """
    Predicts crop yield using the overall trained RandomForestRegressor model.

    Args:
        crop (str): The name of the crop (e.g., 'Rice').
        crop_year (int): The year of the crop.
        season (str): The season (e.g., 'Kharif', 'Whole Year').
        state (str): The state (e.g., 'Assam').
        area (float): The area of cultivation (e.g., 10000.0).
        production (float): The production quantity (e.g., 50000.0).
        annual_rainfall (float): The annual rainfall (e.g., 1500.0).
        fertilizer (float): The fertilizer usage (e.g., 100000.0).
        pesticide (float): The pesticide usage (e.g., 500.0).

    Returns:
        float: The predicted crop yield.
    """
    # Create a DataFrame for the new input, matching the structure of X_train
    input_data = pd.DataFrame([{
        'Crop': crop,
        'Crop_Year': crop_year,
        'Season': season,
        'State': state,
        'Area': area,
        'Production': production,
        'Annual_Rainfall': annual_rainfall,
        'Fertilizer': fertilizer,
        'Pesticide': pesticide
    }])

    # Make prediction using the overall model pipeline
    predicted_yield = model_pipeline.predict(input_data)

    return predicted_yield[0]

### Example Usage of the Prediction Interface

Let's test our `predict_yield_overall_model` function with some sample data. This example uses data similar to an entry from the original dataset.

In [ ]:
# Example prediction
predicted_yield = predict_yield_overall_model(
    crop='Rice',
    crop_year=2010,
    season='Kharif',
    state='West Bengal',
    area=3714.0,
    production=5000.0,
    annual_rainfall=1500.0,
    fertilizer=50000.0,
    pesticide=200.0
)

print(f"Predicted Yield for the given input: {predicted_yield:.2f}")

print("\nNote: This interface uses the *overall* model trained on the entire dataset. If you need state-specific predictions, you would load and use the relevant model from the `state_models` dictionary or the saved `.pkl` files.")

### Using State-Specific Models for Prediction

For more precise predictions, especially for states where the overall model might not perform as well, you can use the individual state models. Here's how you could load a state-specific model and use it for prediction:

First, we need to define a function similar to the `predict_yield_overall_model` but adapted to a state-specific model, which does *not* expect the 'State' column in its input data.

In [ ]:
import os
import pickle

def predict_yield_state_model(state_name, crop, crop_year, season, area, production, annual_rainfall, fertilizer, pesticide):
    """
    Predicts crop yield using a specific state-wise trained model.

    Args:
        state_name (str): The name of the state for which to load the model.
        crop (str): The name of the crop.
        crop_year (int): The year.
        season (str): The season.
        area (float): The area of cultivation.
        production (float): The production quantity.
        annual_rainfall (float): The annual rainfall.
        fertilizer (float): The fertilizer usage.
        pesticide (float): The pesticide usage.

    Returns:
        float: The predicted crop yield, or None if the model cannot be loaded.
    """
    model_path = os.path.join('state_crop_yield_models', f'{state_name.replace(" ", "_").lower()}_crop_yield_model.pkl')

    if not os.path.exists(model_path):
        print(f"Error: Model for {state_name} not found at {model_path}")
        return None

    try:
        with open(model_path, 'rb') as f:
            state_model = pickle.load(f)
    except Exception as e:
        print(f"Error loading model for {state_name}: {e}")
        return None

    # Create a DataFrame for the new input, matching the structure for state-specific models (no 'State' column)
    input_data = pd.DataFrame([{
        'Crop': crop,
        'Crop_Year': crop_year,
        'Season': season,
        'Area': area,
        'Production': production,
        'Annual_Rainfall': annual_rainfall,
        'Fertilizer': fertilizer,
        'Pesticide': pesticide
    }])

    # Make prediction using the state-specific model pipeline
    predicted_yield = state_model.predict(input_data)

    return predicted_yield[0]

### Example Usage of the State-Specific Prediction Interface

Let's use the `predict_yield_state_model` function for a state like 'Jharkhand', which we fine-tuned. Remember that `state_to_tune` still holds 'Jharkhand'.

In [ ]:
# Example prediction for Jharkhand using its specific model
predicted_yield_jharkhand = predict_yield_state_model(
    state_name='Jharkhand',
    crop='Rice',
    crop_year=2010,
    season='Kharif',
    area=3714.0,
    production=5000.0,
    annual_rainfall=1500.0,
    fertilizer=50000.0,
    pesticide=200.0
)

if predicted_yield_jharkhand is not None:
    print(f"Predicted Yield for Jharkhand (using state-specific model): {predicted_yield_jharkhand:.2f}")

### Hyperparameter Tuning for the Overall Model using `RandomizedSearchCV`

To further enhance the performance of our overall crop yield prediction model, we'll perform hyperparameter tuning using `RandomizedSearchCV`. This technique efficiently explores a predefined range of hyperparameter values to find the combination that yields the best performance on our validation data.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

print("\n--- Starting Hyperparameter Tuning for the Overall Model ---")

# Define the parameter distribution for RandomizedSearchCV
# We prefix parameters with 'regressor__' because we are tuning the 'regressor' step in the pipeline
param_dist = {
    'regressor__n_estimators': randint(50, 500), # Number of trees in the forest
    'regressor__max_features': ['sqrt', 'log2', None], # Number of features to consider when looking for the best split
    'regressor__max_depth': randint(5, 30), # Maximum depth of the tree
    'regressor__min_samples_split': randint(2, 20), # Minimum number of samples required to split an internal node
    'regressor__min_samples_leaf': randint(1, 10) # Minimum number of samples required to be at a leaf node
}

# Set up RandomizedSearchCV
# n_iter determines the number of parameter settings that are sampled
random_search = RandomizedSearchCV(estimator=model_pipeline, param_distributions=param_dist,
                                   n_iter=50, cv=5, n_jobs=-1, verbose=2, random_state=20, scoring='r2')

# Perform the random search on the training data
print("  Performing Randomized Search...")
random_search.fit(X_train, y_train)
print("  Randomized Search complete.")

# Get the best parameters and best score
best_params_overall = random_search.best_params_
best_score_overall = random_search.best_score_

print(f"\n  Best parameters found for overall model: {best_params_overall}")
print(f"  Best R-squared from cross-validation for overall model: {best_score_overall:.2f}")

# Evaluate the best model on the test set
best_overall_model = random_search.best_estimator_
y_pred_tuned_overall = best_overall_model.predict(X_test)

mse_tuned_overall = mean_squared_error(y_test, y_pred_tuned_overall)
r2_tuned_overall = r2_score(y_test, y_pred_tuned_overall)

print(f"\n  Fine-tuned Overall Model Evaluation on Test Set:")
print(f"  Mean Squared Error (MSE): {mse_tuned_overall:.2f}")
print(f"  R-squared (R2): {r2_tuned_overall:.2f}")

# Compare with the initial overall model's performance
print(f"\n  Initial Overall Model Evaluation on Test Set (for comparison):")
print(f"  Mean Squared Error (MSE): {mse:.2f}")
print(f"  R-squared (R2): {r2:.2f}")

### Interpretation of Overall Model Tuning Results

After running `RandomizedSearchCV`, we can compare the performance of the fine-tuned overall model against the initial model. The R-squared values and MSE will indicate if the hyperparameter tuning successfully improved the model's predictive capability.

